In [0]:
dbutils.widgets.text("catalog", "global_airline", "Catalog Name")
catalog = dbutils.widgets.get("catalog")

In [0]:
# ---------------------------------------------------------------------------
# Task 5 — Data Quality & Reconciliation Notebook
# Runs AFTER the Gold layer is built.  Raises a pipeline-stopping error
# (AssertionError) when any CRITICAL check fails so that the Databricks Job
# / pipeline run is marked FAILED.
# ---------------------------------------------------------------------------

import pandas as pd
from datetime import datetime

# Catalog / schema — already created, use directly
CATALOG       = f"{catalog}"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA   = "gold"

# Tolerance for reconciliation (≤ 0.01 %)
RECON_TOLERANCE = 0.0001   # 0.01 % expressed as a fraction

# ---------------------------------------------------------------------------
# Results collector
# ---------------------------------------------------------------------------
quality_results = []      # list of dict — one per check
rejected_counts = {}      # per-table rejected-record counts

def _add(check_name, table_name, severity, passed, metric=None, detail=""):
    """Append a row to the results collector."""
    quality_results.append({
        "check_name":     check_name,
        "table_name":     table_name,
        "severity":      severity,        # CRITICAL or WARNING
        "passed":         passed,
        "metric_value":   metric,
        "detail":         detail,
        "run_timestamp":  datetime.utcnow().isoformat(),
    })

def _table_exists(schema, table):
    """Return True if the table exists in Unity Catalog."""
    try:
        spark.table(f"{schema}.{table}").limit(1).collect()
        return True
    except Exception:
        return False

def _count(sql):
    """Return a scalar count from a SQL string."""
    return spark.sql(sql).collect()[0][0]

# ===========================================================================
# CHECK 1 — All eight source tables ingested into Bronze
# ===========================================================================
EXPECTED_SOURCES = [
    "passenger",
    "booking",
    "flight_booking",
    "route",
    "airline_route",
    "airline",
    "nation",
    "region",
]

missing_bronze = []
for src in EXPECTED_SOURCES:
    exists = _table_exists(CATALOG + "." + BRONZE_SCHEMA, src)
    _add(
        f"bronze_ingestion_{src}",
        f"{BRONZE_SCHEMA}.{src}",
        "CRITICAL",
        exists,
        metric=int(exists),
        detail="Table present" if exists else "Table MISSING from Bronze",
    )
    if not exists:
        missing_bronze.append(src)

# ===========================================================================
# CHECK 2 — Primary keys populated and unique (Silver layer)
# ===========================================================================
PK_SPEC = {
    # table_name        : pk column(s)
    "passenger":         ["pa_passengerid"],
    "booking":         ["bk_bookingid"],
    "route":          ["ro_routeid"],
    "airline_route":        ["ar_airlineid", "ar_routeid"],
    "airline":       ["al_airlineid"],
    "nation":          ["n_nationkey"],
    "region":         ["r_regionkey"]
}

for tbl, pk_cols in PK_SPEC.items():
    full_tbl = f"{CATALOG}.{SILVER_SCHEMA}.{tbl}"
    if not _table_exists(f"{CATALOG}.{SILVER_SCHEMA}", tbl):
        _add(f"pk_unique_{tbl}", f"{SILVER_SCHEMA}.{tbl}", "CRITICAL", False,
             detail="Silver table missing — cannot check PK")
        continue

    pk_expr = ", ".join(pk_cols)

    # null count on PK
    null_sql = f"SELECT COUNT(*) FROM {full_tbl} WHERE {pk_cols[0]} IS NULL"
    # if composite key, add OR chains for additional cols
    if len(pk_cols) > 1:
        null_sql = f"SELECT COUNT(*) FROM {full_tbl} WHERE " + " OR ".join(f"{c} IS NULL" for c in pk_cols)
    null_count = _count(null_sql)

    # duplicate count on PK
    dup_sql = (
        f"SELECT COUNT(*) FROM ("
        f"  SELECT {pk_expr}, COUNT(*) AS cnt FROM {full_tbl} "
        f"  GROUP BY {pk_expr} HAVING COUNT(*) > 1"
        f")"
    )
    dup_count = _count(dup_sql)

    pk_ok = (null_count == 0) and (dup_count == 0)
    _add(
        f"pk_unique_{tbl}",
        f"{SILVER_SCHEMA}.{tbl}",
        "CRITICAL",
        pk_ok,
        metric=null_count + dup_count,
        detail=f"PK({pk_expr}) — nulls={null_count}, dups={dup_count}",
    )

# ===========================================================================
# CHECK 3 — Fact table has no null surrogate keys
# ===========================================================================
FACT_TABLE   = f"{CATALOG}.{GOLD_SCHEMA}.fact_bookings"
SURROGATE_KEYS = ["fb_bookingid", "fb_routeid", "fb_airlineid"]

if _table_exists(f"{CATALOG}.{GOLD_SCHEMA}", "fact_bookings"):
    null_sk_sql = (
        f"SELECT COUNT(*) FROM {FACT_TABLE} WHERE "
        + " OR ".join(f"{sk} IS NULL" for sk in SURROGATE_KEYS)
    )
    null_sk_count = _count(null_sk_sql)
    _add(
        "fact_null_surrogate_keys",
        f"{GOLD_SCHEMA}.fact_bookings",
        "CRITICAL",
        null_sk_count == 0,
        metric=null_sk_count,
        detail=f"Rows with null surrogate keys = {null_sk_count}",
    )
else:
    _add("fact_null_surrogate_keys", f"{GOLD_SCHEMA}.fact_bookings",
         "CRITICAL", False, detail="Fact table missing")

# ===========================================================================
# CHECK 4 — Numeric business-rule validation (Gold fact table)
#    ticket_price  > 0
#    discount      ∈ [0, 1]
#    seats         ≥ 1
# ===========================================================================
BR_RULES = [
    ("ticket_price_gt_0", "fb_ticketprice <= 0",                    "fb_ticketprice", "COUNT(*)"),
    ("discount_range_0_1", "fb_discount < 0 OR fb_discount > 1",     "fb_discount",    "COUNT(*)"),
    ("seats_ge_1",        "fb_seats < 1",                          "fb_seats",       "COUNT(*)"),
]

if _table_exists(f"{CATALOG}.{GOLD_SCHEMA}", "fact_bookings"):
    for rule_name, violation_cond, col, _agg in BR_RULES:
        # Guard against missing columns gracefully
        cols = [f.name for f in spark.table(FACT_TABLE).schema.fields]
        if col not in cols:
            _add(f"br_{rule_name}", f"{GOLD_SCHEMA}.fact_bookings",
                 "WARNING", False, detail=f"Column '{col}' not found — skipped")
            continue
        viol_count = _count(f"SELECT COUNT(*) FROM {FACT_TABLE} WHERE {violation_cond}")
        _add(
            f"br_{rule_name}",
            f"{GOLD_SCHEMA}.fact_bookings",
            "CRITICAL",
            viol_count == 0,
            metric=viol_count,
            detail=f"Violations ({violation_cond}) = {viol_count}",
        )
else:
    for rule_name, *_ in BR_RULES:
        _add(f"br_{rule_name}", f"{GOLD_SCHEMA}.fact_bookings",
             "CRITICAL", False, detail="Fact table missing")

# ===========================================================================
# CHECK 5 — Gold ↔ Silver reconciliation (revenue & row counts, tol ≤ 0.01 %)
# ===========================================================================
RECON_PAIRS = [
    # (silver_table, gold_table, revenue_col)
    (f"{CATALOG}.{SILVER_SCHEMA}.flight_booking", FACT_TABLE, "fb_ticketprice * fb_seats"),
]

def _recon(silver_tbl, gold_tbl, revenue_col):
    """Compare row counts and revenue between Silver and Gold."""
    results = []

    s_rows = _count(f"SELECT COUNT(*) FROM {silver_tbl}")
    g_rows = _count(f"SELECT COUNT(*) FROM {gold_tbl}")
    row_diff = abs(s_rows - g_rows)
    row_tol  = max(1, int(s_rows * RECON_TOLERANCE))   # allow at least 1-row diff
    rows_ok = row_diff <= row_tol
    _add(f"recon_rowcount_{gold_tbl.split('.')[-1]}", gold_tbl,
         "CRITICAL", rows_ok,
         metric=row_diff,
         detail=f"Silver rows={s_rows}, Gold rows={g_rows}, diff={row_diff} (tol={row_tol})")
    results.append(rows_ok)

    # revenue comparison
    s_rev = _count(f"SELECT COALESCE(SUM({revenue_col}), 0) FROM {silver_tbl}")
    g_rev = _count(f"SELECT COALESCE(SUM({revenue_col}), 0) FROM {gold_tbl}")
    rev_diff = abs(s_rev - g_rev)
    rev_tol  = max(0.01, float(abs(s_rev)) * RECON_TOLERANCE)
    rev_ok = rev_diff <= rev_tol
    _add(f"recon_revenue_{gold_tbl.split('.')[-1]}", gold_tbl,
         "CRITICAL", rev_ok,
         metric=float(rev_diff),
         detail=f"Silver rev={s_rev}, Gold rev={g_rev}, diff={rev_diff} (tol={rev_tol})")
    results.append(rev_ok)

    return results

for silver_tbl, gold_tbl, rev_col in RECON_PAIRS:
    if _table_exists(*silver_tbl.split(".", 1)) and _table_exists(*gold_tbl.split(".", 1)):
        _recon(silver_tbl, gold_tbl, rev_col)
    else:
        _add(f"recon_{gold_tbl.split('.')[-1]}", gold_tbl,
             "CRITICAL", False, detail="Silver or Gold table missing for reconciliation")

# ===========================================================================
# CHECK 6 — Rejected-record counts per table
# ===========================================================================
# Each Bronze table is expected to have a companion quarantine / rejected
# table named  <source>_rejected  in the Bronze schema.
for src in EXPECTED_SOURCES:
    rej_tbl = f"{CATALOG}.{BRONZE_SCHEMA}.{src}_rejected"
    if _table_exists(f"{CATALOG}.{BRONZE_SCHEMA}", f"{src}_rejected"):
        rej_count = _count(f"SELECT COUNT(*) FROM {rej_tbl}")
    else:
        rej_count = None
    rejected_counts[src] = rej_count
    _add(
        f"rejected_count_{src}",
        f"{BRONZE_SCHEMA}.{src}_rejected",
        "WARNING",                       # informational, not pipeline-stopping
        True,
        metric=rej_count,
        detail=f"Rejected records = {rej_count if rej_count is not None else 'table not found'}",
    )

# ===========================================================================
# REPORT — build a summary DataFrame and persist it
# ===========================================================================
results_df = spark.createDataFrame(pd.DataFrame(quality_results))

# Write to a DQ results table for auditing (create if not exists)
dq_results_table = f"{CATALOG}.{GOLD_SCHEMA}.data_quality_results"
results_df.write.mode("append").saveAsTable(dq_results_table)

# Display full results
print("=" * 80)
print("DATA QUALITY & RECONCILIATION REPORT")
print("=" * 80)
display(results_df.orderBy("severity", "passed"))

print("\nRejected-record counts per table:")
for tbl, cnt in rejected_counts.items():
    print(f"  {tbl:20s}  rejected = {cnt}")

# ===========================================================================
# FAIL THE PIPELINE if any CRITICAL check did not pass
# ===========================================================================
critical_failures = [r for r in quality_results if r["severity"] == "CRITICAL" and not r["passed"]]

if critical_failures:
    print("\n" + "=" * 80)
    print(f"❌ {len(critical_failures)} CRITICAL data-quality check(s) FAILED:")
    print("=" * 80)
    for cf in critical_failures:
        print(f"  [{cf['table_name']}] {cf['check_name']}: {cf['detail']}")
    print("\nPipeline execution will be halted.")
    raise AssertionError(
        f"Data Quality checks FAILED — {len(critical_failures)} critical violation(s). "
        f"See {dq_results_table} for details."
    )
else:
    print("\n" + "=" * 80)
    print("✅ All CRITICAL data-quality checks PASSED.")
    print("=" * 80)